<a href="https://colab.research.google.com/github/santi-boe/Document-Compiler/blob/main/Master_Note_CompilerV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. SETUP & DEPENDENCIES
!pip install PyPDF2 google-api-python-client google-auth-httplib2 google-auth-oauthlib --quiet

import os
import io
import re
import requests
from PyPDF2 import PdfMerger
from google.colab import auth, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# --- 2. THE COMPILER ENGINE ---

def download_as_pdf(service, file_id, output_path):
    """Downloads Google Docs/Slides as PDF or pulls raw PDFs directly."""
    try:
        # Check MIME type to determine if we Export or Download
        meta = service.files().get(fileId=file_id, fields='mimeType', supportsAllDrives=True).execute()
        mime = meta.get('mimeType', '')

        if 'google-apps' in mime:
            # Native Google files need to be exported/converted to PDF
            request = service.files().export_media(fileId=file_id, mimeType='application/pdf')
        else:
            # Existing PDFs are downloaded directly
            request = service.files().get_media(fileId=file_id)

        fh = io.BytesIO()
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()

        with open(output_path, 'wb') as f:
            f.write(fh.getvalue())
        return True
    except Exception as e:
        print(f"  Error processing {file_id}: {e}")
        return False

# --- 3. USER INTERFACE & EXECUTION ---

print("Step 1: Authenticating Google Account...")
auth.authenticate_user()
drive.mount('/content/drive', force_remount=True)

# Generic Output Paths (Sanitized for GitHub)
output_folder = os.path.join('/content/drive/My Drive', 'Colab Notebooks/Compiled_Documents')
temp_dir = '/content/temp_pdfs'
os.makedirs(output_folder, exist_ok=True)
os.makedirs(temp_dir, exist_ok=True)

# Build Service with Discovery URL for better compatibility
service = build('drive', 'v3', discoveryServiceUrl="https://www.googleapis.com/discovery/v1/apis/drive/v3/rest")

master_title = input("\nEnter the Final PDF Title: ")
print("\nPaste your URLs below (one per line). Press Enter TWICE when finished:")

input_urls = []
while True:
    line = input()
    if not line: break
    input_urls.append(line.strip())

pdf_files = []
for i, url in enumerate(input_urls):
    # Extract File ID using Regex
    match = re.search(r'/d/([a-zA-Z0-9-_]+)', url)
    file_id = match.group(1) if match else url

    print(f"Processing {i+1}/{len(input_urls)}: {file_id}...")
    local_path = f'{temp_dir}/file_{i}.pdf'

    if download_as_pdf(service, file_id, local_path):
        # Validate that we actually got a PDF
        if os.path.exists(local_path) and os.path.getsize(local_path) > 100:
            pdf_files.append(local_path)
            print("  Successfully converted!")

# --- 4. FINAL MERGE ---
if pdf_files:
    print(f"\nMerging {len(pdf_files)} files into one Master PDF...")
    merger = PdfMerger()
    for f in pdf_files:
        merger.append(f)

    final_destination = os.path.join(output_folder, f"{master_title}.pdf")
    merger.write(final_destination)
    merger.close()
    print(f"\n✅ SUCCESS! File saved to: {final_destination}")
else:
    print("\n❌ FAILED: No valid files were processed.")